# SadTalker для reels-renderer (GPU Colab)
1) Runtime -> Change runtime type -> T4 GPU
2) Запускайте ячейки по порядку (кнопка ▶)
3) В конце видео скачается на ваш компьютер автоматически.
Ничего копировать из чата не нужно.

In [ ]:
import os
if not os.path.isdir('SadTalker'):
    !git clone -q https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!apt-get -qq install -y ffmpeg
!pip install -q librosa==0.10.2 yacs kornia safetensors pyyaml easydict gfpgan facexlib basicsr
if not os.path.isfile('checkpoints/SadTalker_V0.0.2_256.safetensors'):
    !bash scripts/download_models.sh
    !wget -q https://github.com/Winfredy/SadTalker/releases/download/v0.0.2/BFM_Fitting.zip -O checkpoints/BFM_Fitting.zip
    !cd checkpoints && unzip -oq BFM_Fitting.zip
if not os.path.isfile('examples/avatar.png'):
    !wget -q "https://raw.githubusercontent.com/playalexey-sketch/reels-renderer/arena/019fdb1c-reels-renderer/render/avatar_closed.jpg" -O examples/avatar.png
    !wget -q "https://raw.githubusercontent.com/playalexey-sketch/reels-renderer/arena/019fdb1c-reels-renderer/render/voice5.wav" -O examples/voice5.wav
print('SETUP DONE')

In [ ]:
import numpy as np, sys
try:
    np.VisibleDeprecationWarning = np.exceptions.VisibleDeprecationWarning
except Exception:
    pass
for _a, _v in [('float', float), ('int', int), ('bool', bool), ('complex', complex), ('object', object)]:
    try:
        setattr(np, _a, _v)
    except Exception:
        pass
import torchvision.transforms.functional as _F
import types as _types
_mt = _types.ModuleType('torchvision.transforms.functional_tensor')
_mt.rgb_to_grayscale = _F.rgb_to_grayscale
sys.modules['torchvision.transforms.functional_tensor'] = _mt
import torch
if not getattr(torch.load, '_patched', False):
    _tl = torch.load
    def _nl(*a, **k):
        k['weights_only'] = False
        return _tl(*a, **k)
    _nl._patched = True
    torch.load = _nl
import runpy, sys
sys.argv = ['inference.py', '--driven_audio', 'examples/voice5.wav',
            '--source_image', 'examples/avatar.png',
            '--checkpoint_dir', 'checkpoints', '--result_dir', 'results',
            '--preprocess', 'full', '--enhancer', 'gfpgan',
            '--batch_size', '8', '--size', '256']
runpy.run_path('inference.py', run_name='__main__')
print('RENDER DONE')

In [ ]:
import glob, os
from google.colab import files
vv = sorted(glob.glob('results/**/*.mp4', recursive=True))
print('НАЙДЕНО:', vv)
if not vv:
    print(os.popen('ls -R results 2>/dev/null | head -30').read())
else:
    files.download(vv[-1])